# Charging Station from Final Station
Objective: Place Charging Stations at the last Station of a Line \
Author: Sven Vorgheim \
Disclaimer: The creation of this file was aided by ChatGPT


In [9]:
# Imports
from lxml import etree

In [10]:

from lxml import etree

# Store unique final station IDs
final_station_ids = set()

# Parse XML files
fs_tree = etree.parse("../route_building/filtered_routes.xml")
fs_root = fs_tree.getroot()

bus_stops_tree = etree.parse("../../sumo/berlin_bus_stops.add.xml")
bus_stops_root = bus_stops_tree.getroot()

# Collect unique final station IDs
for route in fs_root.findall("route"):
    stops = route.findall("stop")
    if stops:
        final_station_ids.add(stops[-1].get("busStop"))

# Collect information about those stations
station_values = []

for bus_stop in bus_stops_root.findall("busStop"):
    bus_stop_id = bus_stop.get("id")

    if bus_stop_id in final_station_ids:
        station_values.append({
            "id": bus_stop_id,
            "lane": bus_stop.get("lane"),
            "startPos": bus_stop.get("startPos"),
            "endPos": bus_stop.get("endPos"),
            "name": bus_stop.get("name"),
            "friendlyPos": bus_stop.get("friendlyPos"),
            "parkingLength": bus_stop.get("parkingLength"),
        })

In [ ]:
# Create the root element
additional = etree.Element("additional")

# Create one charging station for each station
for station in station_values:
    etree.SubElement(
        additional,
        "chargingStation",
        id=f"cs_{station['id']}",
        name=f"{station['name']} Charger",
        lane=station["lane"],
        startPos=station["startPos"],
        endPos=station["endPos"],
        power="150000",
        efficiency="0.95",
        chargeInTransit="false",
        friendlyPos="true",
    )

# Write the XML to a file
tree = etree.ElementTree(additional)
tree.write(
    "charging_stations.add.xml",
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
)